## Log and register model: DL PyFunc (torch)

### Purpose
This notebook ingests a deep-learning model (PyTorch) plus metadata trained outside Databricks,
wraps it as a PyFunc with optional preprocessing (e.g. scaler), logs it to MLflow, and registers it in Unity Catalog.

Use this notebook when:
- The customer provides `artifacts.json` and a serialized PyTorch model (`torch_model.pt`)
- You need a custom PyFunc wrapper (e.g. GenericTorchPyFunc) for scaler or I/O handling
- The model class (e.g. SimpleClassifierNet) must match what was used in export/training

This is Stage 2 of the BYOM workflow (Log & Register).

### Expected inputs
Artifacts must already exist in a Unity Catalog volume and follow the expected layout defined in Stage 1.

Required:
- `artifacts.json` (model name, dependencies, feature columns, example input)
- `torch_model.pt` (serialized nn.Module or bundle with model, scaler, meta)

Optional but recommended:
- `checksums.json` for artifact integrity validation


In [ ]:
%pip install torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [ ]:
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("volume_name", "volume", "Volume Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
volume = dbutils.widgets.get("volume_name")

### Load Artifacts from Unity Catalog Volume

Reads model artifacts from the configured UC volume using notebook widgets:

- `catalog_name`
- `schema_name`
- `volume_name`

Ensure these are aligned before running.

If `checksums.json` is present:
- Verify artifact integrity before logging
- Fail fast if mismatches are detected

Recommended for production engagements.


In [ ]:
%run ./00_shared

In [ ]:
import json
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import mlflow
from mlflow import pyfunc
from mlflow.tracking import MlflowClient


# Must match the class used in 0_export_model_and_artifacts; required for torch.load to unpickle torch_model.pt
class SimpleClassifierNet(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hid_dim), nn.ReLU(),
            nn.Linear(hid_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)


# PyFunc wrapper: loads torch_model.pt (nn.Module or {model, scaler, meta}) + artifacts.json; applies optional scaler and returns predictions.
class GenericTorchPyFunc(pyfunc.PythonModel):
    """PyFunc for dl: artifacts.json + torch_model.pt. Pickle may be raw nn.Module or bundle {model, scaler, meta}."""

    def load_context(self, context):
        loaded = torch.load(context.artifacts["torch_model"], map_location="cpu", weights_only=False)
        with open(context.artifacts["artifacts_json"], "r", encoding="utf-8") as f:
            ad = json.load(f)
        if isinstance(loaded, dict) and "model" in loaded and "scaler" in loaded:
            self.model = loaded["model"]
            self.scaler = loaded["scaler"]
            self.feature_columns = loaded["meta"]["feature_columns"]
        else:
            self.model = loaded
            self.scaler = None
            self.feature_columns = ad.get("feature_columns") or [
                f"f{i}" for i in range(len(ad["example_input"][0]))
            ]
        if isinstance(self.model, nn.Module):
            self.model.eval()

    def _pre(self, df: pd.DataFrame) -> torch.Tensor:
        x = df[self.feature_columns].values.astype(np.float32)
        if self.scaler is not None:
            x = self.scaler.transform(x)
        return torch.from_numpy(x)

    def _post(self, raw: torch.Tensor) -> pd.DataFrame:
        y = raw.detach().numpy()
        if y.ndim == 2:
            y = np.argmax(y, axis=1) if y.shape[1] > 1 else y.ravel()
        return pd.DataFrame({"prediction": y.astype(np.float64)})

    def predict(self, context, model_input: pd.DataFrame):
        x = self._pre(model_input)
        with torch.no_grad():
            raw = self.model(x)
        return self._post(raw)


# --- Paths and verification ---
artifact_root = get_artifact_root(catalog, schema, volume)
dl_config = get_dl_config(artifact_root)
checksums_path = get_checksums_path(artifact_root)
verify_artifacts(checksums_path, [("artifacts.json", dl_config["artifacts_json_path"]), ("torch_model.pt", dl_config["torch_model_path"])])
canonical_input, feature_columns = load_canonical_input_from_artifacts_json(dl_config["artifacts_json_path"])


### Infer model signature

Infers input/output schema using canonical sample input (if provided).  
The inferred signature ensures consistent contract enforcement during serving and batch inference.

In [ ]:
# --- Load PyFunc context, run one prediction, infer MLflow signature; model name from artifacts.json ---
dl_artifacts = {"artifacts_json": dl_config["artifacts_json_path"], "torch_model": dl_config["torch_model_path"]}
ctx = SimpleNamespace(artifacts=dl_artifacts, feature_columns=feature_columns)
model = GenericTorchPyFunc()
model.load_context(ctx)
y_pred = model.predict(ctx, canonical_input)
signature = validate_and_infer_signature(canonical_input, y_pred)
with open(dl_config["artifacts_json_path"], "r", encoding="utf-8") as f:
    ad = json.load(f)
model_name = ad.get("model_name", "dl_pyfunc")

### Log to MLflow and register to Unity Catalog

Logs the DL PyFunc model using `mlflow.pyfunc`. Artifacts, signature, and metadata are recorded in MLflow tracking.

Registers the logged model in Unity Catalog Model Registry.

Ensures:
- Governance
- Lineage
- Versioning

In [ ]:
# --- Log as pyfunc (artifacts copied into run), register in Unity Catalog ---
mlflow.set_registry_uri("databricks-uc")
deps = ad.get("dependencies", ["torch", "pandas", "numpy", "scikit-learn", "joblib"])
with mlflow.start_run() as run:
    mlflow.pyfunc.log_model(python_model=model, artifact_path=model_name, artifacts=dl_artifacts, input_example=canonical_input, signature=signature, extra_pip_requirements=deps)
    model_uri = f"runs:/{run.info.run_id}/{model_name}"
registered_name = f"{catalog}.{schema}.{model_name}"
result = mlflow.register_model(model_uri=model_uri, name=registered_name)
client = MlflowClient()
register_in_uc(client, model_uri, registered_name, str(result.version))

# For job pipelines: pass version to downstream tasks (e.g. evaluation)
dbutils.jobs.taskValues.set(key="model_version", value=str(result.version))
dbutils.jobs.taskValues.set(key="model_name", value=registered_name)

# Sanity check: load the version we just registered and run predict (Champion is set later in 3_model_approval)
loaded = mlflow.pyfunc.load_model(f"models:/{registered_name}/{result.version}")
loaded.predict(canonical_input)
print(f"{registered_name} v{result.version} loaded; predict OK.")